In [1]:
import pathlib

import yaml
import pandas as pd

import jax.random as jr

from torch.utils.data import DataLoader

import altair as alt
import plotly.express as px
import plotly.graph_objects as go

import trapallada.data
import trapallada.model.signal
import trapallada.model.image
import trapallada.util
import trapallada.core
import trapallada.plot

# import ipywidgets as widgets

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


So that `import`ed Python modules are automatically reloaded when modified

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Input Files

Relative path and *Base* file name for the results (trained model). It should be a `str`.

In [4]:
# input_files_stem = '../results/vanilla_signals/res_good' # <---- good
input_files_stem = '../results/res_good' # <---- good

It is turned into a `Pathlib`

In [5]:
input_files_stem = pathlib.Path(input_files_stem)
input_files_stem

PosixPath('../results/res_good')

In [6]:
# w_file_upload = widgets.FileUpload(
#     accept='.eqx',
#     multiple=False
# )
# display(w_file_upload)

In [7]:
# w_file_upload.value

## Model

From the above *stem*, several file names are derived

In [8]:
# parameters_file = pathlib.Path('parameters.yaml')
parameters_file = input_files_stem.with_suffix('.yaml')
assert parameters_file.exists()

# weights_file = pathlib.Path('foo.eqx')
weights_file = input_files_stem.with_suffix('.eqx')
assert weights_file.exists()

results_file = input_files_stem.with_suffix('.parquet')
assert results_file.exists()

In [9]:
with parameters_file.open() as f:
        parameters = yaml.safe_load(f)
# %pycat {parameters_file}
%cat {parameters_file}

PRNG seeds:
  model initialization:
    decoder: 43
    encoder: 42
  sampling:
    training set: 14
    validation set: 15
  training dataloader: 43
  training-validation splitting: 42
dataset:
  files:
    metadata_file: Sync/git/trapallada/data/vanilla_metadata.parquet
    signals_file: Sync/git/trapallada/data/vanilla_signals.pt
    signals_hyperparameters_file: Sync/git/trapallada/data/vanilla_hyperparameters.yaml
  hyperparameters:
    bin_size: 63
    max_n_sources: 4
    min_power: 0.0
    step: 10
    window_size: 225
encoder:
  input_size:
  - 1
  - 225
loss:
  x_sd: 0.25
model:
  channels:
  - 12
  - 12
  - 12
  dim_z: 50
  kernel_sizes:
  - 64
  - 32
  - 16
  stride: 2
training:
  batch_size: 32
  dl_rng_key: 43
  learning_rate: 0.001
  n_epochs: 5000
  train_test_split:
    test_size: 0.2


## Data

The files making up the dataset used to train the model...

In [10]:
signals_file = pathlib.Path.home() / parameters['dataset']['files']['signals_file']
metadata_file = pathlib.Path.home() / parameters['dataset']['files']['metadata_file']

...are loaded...

In [ ]:
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, normalize=True)
# len(ds)

...along with the metrics from the training process

In [ ]:
results_df = pd.read_parquet(results_file)
results_df

# Training curves

In [ ]:
def metric_chart(df: pd.DataFrame, metric: str, n_rows: int = 4_000):

    sub_df = df.loc[:n_rows,[('training', metric), ('validation', metric)]]
    sub_df.columns = sub_df.columns.get_level_values(0)
    chart = alt.Chart(
        sub_df.reset_index()
    ).transform_fold(
        sub_df.columns.to_list(), ['dataset', metric]
    ).mark_line().encode(
        x='index:Q', y=f'{metric}:Q', color='dataset:N'
    ).properties(width=600, height=400)

    return chart

metric_chart(results_df, 'loss') | metric_chart(results_df, 'loglikelihood') | metric_chart(results_df, 'kl')

In [14]:
model = trapallada.model.signal.load_vae(weights_file, parameters_file)

Number of parameters

In [ ]:
sum(trapallada.util.get_layers_n_parameters(model).values())

Data loaders for both the training and validation sets

In [16]:
(train_dl, validation_dl), (train_ds, validation_ds) = trapallada.data.dataloaders_from_dataset(
    ds,
    splitting_parameters=parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    batch_size=parameters['training']['batch_size'], batching_random_state=parameters['PRNG seeds']['training dataloader'], return_datasets=True)

A batch from the **validation** set

In [ ]:
# iter_dl = iter(train_dl) # training
iter_dl = iter(validation_dl) # validation
next(iter_dl), next(iter_dl)
batch = next(iter_dl)
batch[1].shape

What are the different labels, *if any* (useful to check whether every class is present)?

In [ ]:
batch[2].unique()

An element from the above batch is encoded, decoded, and a prediction plotted

In [ ]:
# i = 0
# i = 1
i = 31

# indexes from the above batch
i_s, *_ = batch

def plot_prediction(ds, i):
    
    x = ds[i][1].numpy()
    x_pred, z_mean, z_sd = model(x, jr.PRNGKey(42))
    display(ds.metadata_df.loc[i])
    df = pd.DataFrame({'prediction': x_pred[0], 'actual': x.ravel()})
    n_rows = 2000
    
    return trapallada.plot.df(df.iloc[:n_rows])

# plot_prediction(ds, 25)
plot_prediction(ds, i_s[i].item())

## Javi's paper

A function that looks up a particular source (a `FTZ` file) and plots the requested window along with its reconstruction

In [20]:
def plot_source_window(source: str, i_window: int = -1):

    df = ds.metadata_df[ds.metadata_df['source'] == source]

    # display((df['end'] - df['start']).tail())

    try:
        series = df.iloc[i_window]
    
    except IndexError:

        print('not found')

        return None
    
    index = series.name

    display(plot_prediction(ds, index))
    
    # print(f'seen during training? {index in train_ds[:][0]}' )

    return series

In [ ]:
javi_obs = plot_source_window('P0605950101PNS003SRCTSR8001.FTZ', -1)
javi_meta = [javi_obs]

In [ ]:
# javi_meta_2 = plot_source_window('P0605950101PNS003SRCTSR8003.FTZ')
javi_obs = plot_source_window('P0605950101PNS003SRCTSR8003.FTZ')
javi_meta.append(javi_obs)

In [ ]:
javi_obs = plot_source_window('P0605950101PNS003SRCTSR8005.FTZ')
javi_meta.append(javi_obs)

In [ ]:
javi_obs = plot_source_window('P0605950101PNS003SRCTSR8007.FTZ')
javi_meta.append(javi_obs)

In [ ]:
i_javi = [e.name for e in javi_meta if e is not None]
i_javi

# Latent space projection

## 2D

(Out of the full dataset), the curves in the given `DataLoader` are encoded into the latent space (last columns in the resulting `DataFrame`)

In [ ]:
# fulldataset_dl = DataLoader(ds, batch_size=parameters['training']['batch_size'], shuffle=False)
fulldataset_dl = DataLoader(ds, batch_size=128, shuffle=False)
proj_df = trapallada.util.make_projections_df(ds, fulldataset_dl, model.encoder)
proj_df.loc[train_ds[:][0], 'set'] = 'training'
proj_df.loc[validation_ds[:][0], 'set'] = 'validation'
proj_df

Embedded space is close to a standard Gaussian?

In [ ]:
proj_df.filter(like='z_').agg(['mean', 'std'])

...Javi's paper

In [ ]:
proj_df.loc[i_javi, ['tsne_1', 'tsne_2', 'set']]

In [ ]:
fig = px.scatter(proj_df.reset_index(), x='tsne_1', y='tsne_2', color='tar file', symbol='set', hover_data=['index', 'source'], opacity=0.5, width=1_200, height=800)
fig.add_trace(go.Scatter(
    x=proj_df.loc[i_javi, 'tsne_1'].tolist(), y=proj_df.loc[i_javi, 'tsne_2'].tolist(), mode='markers', marker=dict(size=20, symbol='star', color='blue')
))
fig.show()

In [ ]:
# ds[2239]
# plot_prediction(ds, 2239)
# plot_prediction(ds, 12348)
# plot_prediction(ds, 10480)
# plot_prediction(ds, 5778)
# plot_prediction(ds, 3995)
# plot_prediction(ds, 7102)
# plot_prediction(ds, 10190)
# ------------- (by looking at above 3D picture)
# plot_prediction(ds, 33170)
# plot_prediction(ds, 10868)
# -------------
# plot_prediction(ds, 54087)
# plot_prediction(ds, 36097)
# plot_prediction(ds, 2673)
# plot_prediction(ds, 6004)
# plot_prediction(ds, 16900)
# plot_prediction(ds, 27243)
# plot_prediction(ds, 50287)
# plot_prediction(ds, 49489)
# plot_prediction(ds, 11717) # <---- noise
# plot_prediction(ds, 16903)
plot_prediction(ds, 12455)

## 3D

The latent space is projected into a 3D space

In [ ]:
proj3D_df = trapallada.util.make_projections_df(ds, fulldataset_dl, model.encoder, n_projected_components=3)
proj3D_df.loc[train_ds[:][0], 'set'] = 'training'
proj3D_df.loc[validation_ds[:][0], 'set'] = 'validation'
proj3D_df.head()

In [ ]:
proj3D_df.loc[i_javi].filter(like='tsne_')

In [ ]:
fig = px.scatter_3d(proj3D_df.reset_index(), x='tsne_1', y='tsne_2', z='tsne_3', hover_data=['index'], color='tar file', opacity=0.5, width=1_200, height=800)
fig.add_trace(go.Scatter3d(
    x=proj3D_df.loc[i_javi, 'tsne_1'].tolist(), y=proj3D_df.loc[i_javi, 'tsne_2'].tolist(), z=proj3D_df.loc[i_javi, 'tsne_3'].tolist(), mode='markers', marker=dict(size=12, symbol='diamond', color='blue')
))
fig.show()